In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.utils import to_categorical
import tarfile
import pickle
import numpy as np

In [ ]:


def load_cifar10(tar_path='cifar-10-batches-py.tar.gz'):
    with tarfile.open(tar_path, 'r:gz') as tar:
        def get_batch(name):
            member = [m for m in tar.getmembers() if m.name.endswith(name)][0]
            return pickle.load(tar.extractfile(member), encoding='bytes')

        # Load 5 training batches
        x_tr, y_tr = [], []
        for i in range(1, 6):
            b = get_batch(f'data_batch_{i}')
            x_tr.append(b[b'data'])
            y_tr.extend(b[b'labels'])

        # Load test batch
        b_test = get_batch('test_batch')
        x_te, y_te = b_test[b'data'], b_test[b'labels']

    # Reshape from flat vectors (3072,) -> (32, 32, 3) image format
    X_train = np.vstack(x_tr).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_train = np.array(y_tr).reshape(-1, 1)

    X_test = x_te.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_test = np.array(y_te).reshape(-1, 1)

    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = load_cifar10('cifar-10-python.tar.gz')

In [ ]:
#step 2 - tf.data pipeline: resize to 224x224 one batch at a time (keeps X_train/X_test as 32x32, saves RAM)
BATCH = 64

def prep(img, label):
    img = tf.image.resize(img, (224, 224))  # stays float, 0-255 (Rescaling layer in the model handles scaling)
    return img, label[0]                    # y is (N,1) -> scalar int label

def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(10000)
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, shuffle=True)
test_ds = make_ds(X_test, y_test)

In [ ]:
#step 3 - Load MobileNetV2 without top classification head
from tensorflow.keras.applications import MobileNetV2

#load MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet',
                         include_top=False, #without top classification head
                         input_shape=(224, 224, 3))

base_model.trainable = False  #going with freezed pretrained layers (for now)

In [ ]:
#step 4 - classification head. Only quantizeml-supported layers: Rescaling, GAP, Dense, Dropout.
#No softmax (not supported) -> the model outputs logits and the loss uses from_logits=True.
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1/127.5, offset=-1, input_shape=(224, 224, 3)),  # raw 0-255 -> [-1, 1] inside the model
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10),
])
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
#step 5 - train the head (base frozen)
history = model.fit(train_ds, validation_data=test_ds, epochs=10)
print(model.evaluate(test_ds))

In [ ]:
#step 6 - fine-tune (optional): unfreeze the base, recompile with a low learning rate
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
history_fine = model.fit(train_ds, validation_data=test_ds, epochs=5)
print(model.evaluate(test_ds))

In [ ]:
#calibration samples for quantization: 16 batches of 64 = 1024 raw 0-255 images
samples = np.concatenate([x.numpy() for x, _ in train_ds.take(16)]).astype(np.float32)

In [ ]:
import tf_keras as keras
from tf_keras.applications import MobileNetV2

inputs = keras.Input((224, 224, 3))
x = model.layers[0](inputs)                                    # Rescaling
base = MobileNetV2(include_top=False, weights=None, input_tensor=x)   # flat layers
base.set_weights(model.layers[1].get_weights())                # copy trained weights
x = base.output
for layer in model.layers[2:]:                                 # GAP, Dense, Dropout, Dense
    x = layer(x)
flat_model = keras.Model(inputs, x)

print([type(l).__name__ for l in flat_model.layers[:4]])

In [ ]:
from quantizeml.models import QuantizationParams
qparams4 = QuantizationParams(input_weight_bits=8, weight_bits=8, activation_bits=4)

from quantizeml.models import quantize
quantized_model = quantize(flat_model, qparams=qparams4, samples=samples)
from cnn2snn import convert
akida_model = convert(quantized_model)

In [ ]:
import numpy as np
import akida

# Conversion check, then accuracy of float vs quantized vs Akida on 2000 test images.
akida_model.summary()
print("input shape :", akida_model.input_shape)
print("output shape:", akida_model.output_shape)

xs, ys = [], []
for xb, yb in test_ds.take(32):  # 32 batches x 64 = 2048
    xs.append(xb.numpy()); ys.append(yb.numpy())
x = np.concatenate(xs)[:2000].astype(np.float32)
y = np.concatenate(ys)[:2000]

float_pred = model.predict(x, verbose=0).argmax(axis=1)
quant_pred = quantized_model.predict(x, verbose=0).argmax(axis=1)
akida_pred = np.asarray(akida_model.forward(x.astype(np.uint8))).reshape(len(x), -1).argmax(axis=1)
print("float    :", np.mean(float_pred == y))
print("quantized:", np.mean(quant_pred == y))
print("akida    :", np.mean(akida_pred == y))
print("akida vs quantized agreement:", np.mean(akida_pred == quant_pred))

print("devices:", akida.devices())

In [ ]:
model.save("mobilenetv2_cifar10.h5")

In [ ]:
model.save('my_model.keras')